In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")

In [2]:
orders_clean = orders.copy()
payments_clean = payments.copy()

payment_order_ids = set(payments_clean["order_id"])

orphan_orders = orders_clean[
    ~orders_clean["order_id"].isin(payment_order_ids)
]

print(f"Orphan orders: {len(orphan_orders)}")

Orphan orders: 1


In [3]:
orders_clean = orders_clean[
    orders_clean["order_id"].isin(payment_order_ids)
].copy()

In [4]:
orders_clean["order_delivered_customer_date"] = pd.to_datetime(
    orders_clean["order_delivered_customer_date"]
)

orders_clean["order_estimated_delivery_date"] = pd.to_datetime(
    orders_clean["order_estimated_delivery_date"]
)

missing_delivery = (
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].isna())
)

print("Missing delivered dates:", missing_delivery.sum())

Missing delivered dates: 8


In [5]:
orders_clean["order_delivered_customer_date_imputed"] = (
    orders_clean["order_delivered_customer_date"]
    .fillna(orders_clean["order_estimated_delivery_date"])
)

In [6]:
print("Payments with payment_value <= 0:",
      (payments_clean["payment_value"] <= 0).sum())

payments_clean = payments_clean[
    payments_clean["payment_value"] > 0
].copy()

Payments with payment_value <= 0: 9


In [7]:
print("=== DATA QUALITY CHECK ===")

print(
    "Orphan orders:",
    len(orders_clean[
        ~orders_clean["order_id"].isin(payments_clean["order_id"])
    ])
)

print(
    "Delivered orders with missing delivery date:",
    (
        (orders_clean["order_status"] == "delivered") &
        (orders_clean["order_delivered_customer_date"].isna())
    ).sum()
)

print(
    "Payments <= 0:",
    (payments_clean["payment_value"] <= 0).sum()
)

=== DATA QUALITY CHECK ===
Orphan orders: 3
Delivered orders with missing delivery date: 8
Payments <= 0: 0
